## 프로젝트 실행 순서 

In [ ]:
#1단계 

python scripts/build_yolodataset.py 

# raw_data/ 안의 이미지 + JSON 어노테이션을 YOLO 학습 형식으로 바꿉니다. 

#2단계

python scripts/train_yolo.py --data output/dataset/data.yaml

# 1단계에서 만든 data.yaml로 YOLOv8을 학습해, 이미지에서 동물 bbox를 탐지하는 모델을 만듭니다.

# 3단계: CNN용 crop 데이터셋 만들기

python scripts/build_cnn_crops.py

# output/dataset의 train/val 이미지 + YOLO 라벨을 이용해, 정답 bbox를 잘라 CNN 학습용 이미지를 만듭니다.

# 4단계: non_animal(오탐) crop 수집

python scripts/collect_non_animal_crops.py

# 2단계에서 학습한 YOLO 모델로 train/val 이미지에 추론을 돌리고, GT와 겹치지 않는 예측(오탐)을 crop해서 Stage1의 non_animal로 씁니다.

# 5단계: CNN Stage1 학습 (오탐 제거)

python scripts/train_cnn.py --stage 1


# Stage1 데이터(animal/ vs non_animal/)로 이진 분류 모델을 학습합니다.
# “동물 vs 비동물(오탐)”을 구분해, 추론 시 YOLO가 잡은 박스 중 진짜 동물만 남기기 위한 모델입니다.

# 6단계: CNN Stage2 학습 (종 분류)


python scripts/train_cnn.py --stage 2

# Stage2 데이터(클래스별 폴더)로 8종 다중 분류 모델을 학습합니다.

7단계: 2단계 추론 (실제 사용)

python scripts/run_two_stage_inference.py <이미지_또는_폴더_경로>